# Bronze — Yahoo pull log

`landing.yf_pull_log_raw` → `bronze.yf_pull_log`. Every column cast to STRING.

This is the one table the pipeline writes about itself rather than receiving from a
source, so skipping Bronze could be argued for it. It is not skipped: Silver then reads
from exactly one layer, and "every Landing table has a Bronze twin" is a rule with no
exceptions to explain.

The cost is visible here — `row_count` and `pulled_at` are already correctly typed and
get stringified anyway, for Silver to cast straight back.

Expected: **122 rows**, matching Landing exactly.

In [0]:
CATALOG = "`index-vs-trust-pipeline`"
SOURCE = f"{CATALOG}.landing.yf_pull_log_raw"
TARGET = f"{CATALOG}.bronze.yf_pull_log"

In [0]:
src_columns = spark.table(SOURCE).columns

# Cast whatever arrived. Naming columns in advance would silently drop any the source
# adds, which is the one thing this layer must never do.
cast_list = ",\n  ".join(f"CAST(`{c}` AS STRING) AS `{c}`" for c in src_columns)
sql = f"CREATE OR REPLACE TABLE {TARGET} AS\nSELECT\n  {cast_list}\nFROM {SOURCE}"

print(f"{len(src_columns)} columns found: {src_columns}\n")
print(sql)

spark.sql(sql)
print(f"\nwrote {TARGET}")

## Verification

In [0]:
%sql
-- Row parity, plus the refusals that carry the survivorship argument.
SELECT
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.landing.yf_pull_log_raw) AS landing_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.yf_pull_log)      AS bronze_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
    WHERE status = 'NODATA')                                               AS nodata_rows;

Expect **122 / 122 / 19** — 18 trusts Yahoo will not serve, plus SPLG.

Those 19 rows are the only record in the warehouse that those companies were ever asked
for. If this number drops, the survivorship evidence has been lost somewhere.

In [0]:
%sql
-- The contract. Any non-STRING column here is a bug.
SELECT COUNT(*)                                              AS columns_total,
       SUM(CASE WHEN data_type <> 'STRING' THEN 1 ELSE 0 END) AS not_string
FROM `index-vs-trust-pipeline`.information_schema.columns
WHERE table_schema = 'bronze' AND table_name = 'yf_pull_log';

Expect **10 columns, 0 not_string** — including `row_count` and `pulled_at`, which were
INT and TIMESTAMP in Landing.